In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import os

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Load data and model
test = pd.read_csv("dataset/test_data.csv")

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11176.57it/s]


In [5]:
embeddings = model.encode(test["text"].tolist(), normalize_embeddings=True)

embeddings

array([[ 0.00390542,  0.07723191, -0.07646008, ...,  0.05257715,
        -0.00114121,  0.02817407],
       [-0.03732254, -0.03469764, -0.006497  , ..., -0.06907201,
         0.04088127,  0.0425351 ],
       [ 0.01966873,  0.03285254, -0.01347798, ...,  0.01995277,
         0.04706172, -0.03669468],
       ...,
       [-0.05996673, -0.02826674, -0.03009571, ..., -0.05155658,
         0.05571827,  0.05837261],
       [-0.07406387, -0.04858521, -0.02400332, ...,  0.00594697,
         0.01343528, -0.0524881 ],
       [ 0.0060335 ,  0.00922063, -0.04258459, ..., -0.09076742,
        -0.00935875,  0.04864402]], shape=(7000, 384), dtype=float32)

In [6]:
movie_concept = model.encode(["movie review film acting director cinema plot"])
news_concept = model.encode(["breaking news politics world finance sports technology business local events"])

In [7]:
sim_movie = movie_concept @ embeddings.T
sim_news = news_concept @ embeddings.T

label = []

sim_movie = sim_movie.squeeze(0)
sim_news = sim_news.squeeze(0)

for i in range(len(sim_movie)):
    label.append(1 if sim_movie[i] > sim_news[i] + 0.06 else 0)

In [ ]:
from sklearn.cluster import KMeans

df = pd.DataFrame({
    "embeddings": embeddings.tolist(),
    "noise": label
})

hdb = KMeans(n_clusters=4)
preds = hdb.fit_predict(embeddings)

In [70]:
output = pd.concat([pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test["datapointID"],
    "answer": label
}), pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test["datapointID"],
    "answer": preds
})], ignore_index=True)

output.to_csv("output.csv", index=False)